# 🛡️ GaudOn: URL Classifier Training (V4 - Corrected)

**The Problem:** Overfitting on exact URL metadata (length/https).

**The Fix:** Strictly limited model depth and augmented benign data.

In [ ]:
!pip install scikit-learn joblib python-whois tldextract python-Levenshtein httpx

In [ ]:
import httpx, pandas as pd, zipfile, io, os, re, random, math, tldextract
from urllib.parse import urlparse

print('📥 Gathering Data...')
malicious_urls = []
try:
    with httpx.Client(follow_redirects=True) as client:
        malicious_urls.extend(client.get('https://openphish.com/feed.txt').text.strip().split('\n'))
        res = client.get('https://urlhaus.abuse.ch/downloads/csv_recent/')
        lines = [line for line in res.text.splitlines() if not line.startswith('#')]
        df = pd.read_csv(io.StringIO('\n'.join(lines)), names=['id','date','url','status','last','threat','tags','link','rep'])
        malicious_urls.extend(df['url'].tolist())
except: pass
malicious_urls = list(set(malicious_urls))

benign_urls = []
try:
    res = httpx.get('https://tranco-list.eu/top-1m.csv.zip')
    with zipfile.ZipFile(io.BytesIO(res.content)) as z:
        with z.open('top-1m.csv') as f:
            df_b = pd.read_csv(f, names=['rank', 'domain'])
            subset = df_b['domain'].iloc[:len(malicious_urls)].tolist()
            paths = ['/', '/home', '/login', '/about', '/api/v1']
            for d in subset:
                url = f'https://{d}{random.choice(paths)}'
                if random.random() > 0.5: url = url.replace('https://', 'https://www.')
                benign_urls.append(url)
except: pass
print(f'✅ {len(malicious_urls)} Malicious | {len(benign_urls)} Benign')

In [ ]:
def calc_entropy(s):
    if not s: return 0.0
    p = [float(s.count(c))/len(s) for c in dict.fromkeys(list(s))]
    return -sum([i * math.log(i)/math.log(2.0) for i in p])

def extract_features(u):
    try:
        u = u.strip().rstrip('/')
        p = urlparse(u)
        ext = tldextract.extract(u)
        d = ext.domain
        sub = ext.subdomain.replace('www','').strip('.')
        sub_d = len(sub.split('.')) if sub else 0
        risky = ['.xyz','.tk','.top','.ml','.ga']
        tld_s = 0.8 if any(ext.suffix.endswith(t[1:]) for t in risky) else 0.1
        look = ['0','1','3','5']
        v = [0, sum(1 for k in ['login','verify','secure'] if k in u.lower()), calc_entropy(d), sub_d, 1 if u.startswith('https') else 0, len(u), d.count('-'), tld_s, sum(u.count(c) for c in ['@','=','?']), sum(d.count(n) for n in look)/len(d) if d else 0, u.count('%'), 1 if '//' in u[u.find('//')+2:] else 0, 1 if re.match(r'^\d', d) else 0, (sum(1 for c in d.lower() if c in 'aeiou')/sum(1 for c in d.lower() if c.isalpha() and c not in 'aeiou')) if any(c.isalpha() and c not in 'aeiou' for c in d) else 0, 0, 1 if f'{ext.domain}.{ext.suffix}' in ['bit.ly','t.co'] else 0, 1 if p.port and p.port not in [80,443] else 0]
        return v
    except: return None

print('⚙️ Extracting...')
data, labels = [], []
for u in malicious_urls: 
    f = extract_features(u)
    if f: data.append(f); labels.append(1)
for u in benign_urls: 
    f = extract_features(u)
    if f: data.append(f); labels.append(0)
print(f'✅ Dataset: {len(data)}')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import numpy as np, joblib

X, y = np.array(data), np.array(labels)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_leaf=10, random_state=42)
clf.fit(X_train, y_train)
print(f'Accuracy: {clf.score(X_test, y_test):.4f}')

for t in ['https://www.google.com', 'https://gtbank.com']:
    p = clf.predict_proba([extract_features(t)])[0][1]
    print(f'[{t}]: Phish Prob = {p:.4f}')

joblib.dump(clf, 'model.joblib')
print('✅ Download model.joblib')